# Lab 2: TF-IDF Normalization Combinations\n\nTesting 6 combinations of TF and IDF normalizations on IndicCorpV2 language identification.

In [1]:
import sys
import re
import math
import random
import numpy as np
from collections import Counter
from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print('Imports OK.')


Imports OK.


In [2]:
import os, pickle
CACHE_FILE = os.path.join(os.getcwd(), 'dataset_sentences.pkl')

if not os.path.exists(CACHE_FILE):
    raise FileNotFoundError("dataset_sentences.pkl not found! Run lab1.ipynb first to generate it.")

with open(CACHE_FILE, 'rb') as f:
    dataset_sentences = pickle.load(f)

print(f"Loaded {len(dataset_sentences)} languages from cache.")

# We only need the keys for LANGUAGE_SPLITS
LANGUAGE_SPLITS = list(dataset_sentences.keys())


Loaded 23 languages from cache.


In [3]:
def stratified_split(dataset_sentences, train_r=0.8, val_r=0.1, test_r=0.1, seed=42):
    """Stratified split: equal per-language proportion in each set."""
    assert abs(train_r + val_r + test_r - 1.0) < 1e-9, 'ratios must sum to 1'
    rng = random.Random(seed)
    train_texts, train_labels = [], []
    val_texts,   val_labels   = [], []
    test_texts,  test_labels  = [], []
    for lang, sents in dataset_sentences.items():
        s = sents[:]
        rng.shuffle(s)
        n = len(s)
        nt = int(n * train_r)
        nv = int(n * val_r)
        train_texts.extend(s[:nt]);        train_labels.extend([lang]*nt)
        val_texts.extend(s[nt:nt+nv]);     val_labels.extend([lang]*nv)
        test_texts.extend(s[nt+nv:]);      test_labels.extend([lang]*(n-nt-nv))
    return train_texts, train_labels, val_texts, val_labels, test_texts, test_labels


train_texts, train_labels, val_texts, val_labels, test_texts, test_labels = \
    stratified_split(dataset_sentences, seed=SEED)

print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")

train_dist = Counter(train_labels)
val_dist   = Counter(val_labels)
test_dist  = Counter(test_labels)
print()
print(f"{'Language':<15} {'Train':>6} {'Val':>6} {'Test':>6}")
print('-' * 38)
for lang in LANGUAGE_SPLITS:
    print(f"{lang:<15} {train_dist.get(lang,0):>6} {val_dist.get(lang,0):>6} {test_dist.get(lang,0):>6}")


Train: 18032 | Val: 2254 | Test: 2277

Language         Train    Val   Test
--------------------------------------
asm_Beng           784     98     99
ben_Beng           784     98     99
brx_Deva           784     98     99
doi_Deva           784     98     99
gom_Deva           784     98     99
guj_Gujr           784     98     99
hin_Deva           784     98     99
kan_Knda           784     98     99
kas_Arab           784     98     99
mai_Deva           784     98     99
mal_Mlym           784     98     99
mar_Deva           784     98     99
mni_Mtei           784     98     99
npi_Deva           784     98     99
ory_Orya           784     98     99
pan_Guru           784     98     99
san_Deva           784     98     99
snd_Deva           784     98     99
tam_Taml           784     98     99
tel_Telu           784     98     99
urd_Arab           784     98     99
khasi              784     98     99
santhali           784     98     99


In [4]:
unique_labels = sorted(set(train_labels))
label2idx = {lbl: i for i, lbl in enumerate(unique_labels)}
idx2label = {i: lbl for lbl, i in label2idx.items()}
NUM_CLASSES = len(unique_labels)

print(f"Number of classes: {NUM_CLASSES}")
for lbl, idx in label2idx.items():
    print(f"  {lbl} -> {idx}")

y_train = np.array([label2idx[l] for l in train_labels])
y_val   = np.array([label2idx[l] for l in val_labels])
y_test  = np.array([label2idx[l] for l in test_labels])


Number of classes: 23
  asm_Beng -> 0
  ben_Beng -> 1
  brx_Deva -> 2
  doi_Deva -> 3
  gom_Deva -> 4
  guj_Gujr -> 5
  hin_Deva -> 6
  kan_Knda -> 7
  kas_Arab -> 8
  khasi -> 9
  mai_Deva -> 10
  mal_Mlym -> 11
  mar_Deva -> 12
  mni_Mtei -> 13
  npi_Deva -> 14
  ory_Orya -> 15
  pan_Guru -> 16
  san_Deva -> 17
  santhali -> 18
  snd_Deva -> 19
  tam_Taml -> 20
  tel_Telu -> 21
  urd_Arab -> 22


In [5]:
class CustomTFIDFVectorizer:
    """
    Custom TF-IDF vectorizer supporting 6 combinations of TF and IDF normalizations.

    tf_type:
      - 'unnormalized': 1 + log(1 + raw_freq)
      - 'norm_total_words': 1 + log(1 + (raw_freq / total_words))
      - 'norm_max_freq': 1 + log(1 + (raw_freq / max_freq))

    idf_type:
      - 'unnormalized': 1 + log((N + 1) / (df + 1))
      - 'norm_max_df': 1 + log((max_df + 1) / (df + 1))
    """

    def __init__(self, tf_type='unnormalized', idf_type='unnormalized', max_features=25000, min_df=2):
        self.tf_type = tf_type
        self.idf_type = idf_type
        self.max_features = max_features
        self.min_df = min_df

        self.vocabulary_ = {}
        self.idf_values_ = {}
        self.vocab_list_ = []

    def _word_ngrams(self, text, n):
        tokens = text.split()
        if len(tokens) < n:
            return []
        return ['W:' + ' '.join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

    def _char_ngrams(self, text, n):
        text = text.strip()
        if len(text) < n:
            return []
        return [f'C{n}:' + text[i:i+n] for i in range(len(text)-n+1)]

    def _extract_features(self, text):
        feats = []
        feats.extend(self._word_ngrams(text, 1))
        feats.extend(self._word_ngrams(text, 2))
        feats.extend(self._char_ngrams(text, 2))
        feats.extend(self._char_ngrams(text, 3))
        feats.extend(self._char_ngrams(text, 4))
        return feats

    def _compute_tf(self, features):
        if not features:
            return {}

        counts = Counter(features)
        total_words = len(features)
        max_freq = max(counts.values())

        tf = {}
        for t, raw_freq in counts.items():
            if self.tf_type == 'unnormalized':
                tf[t] = 1 + math.log(1 + raw_freq)
            elif self.tf_type == 'norm_total_words':
                tf[t] = 1 + math.log(1 + (raw_freq / total_words))
            elif self.tf_type == 'norm_max_freq':
                tf[t] = 1 + math.log(1 + (raw_freq / max_freq))
            else:
                raise ValueError(f"Unknown tf_type: {self.tf_type}")
        return tf

    def fit(self, texts):
        N = len(texts)
        all_features = []
        for text in texts:
            all_features.append(self._extract_features(text))

        df = Counter()
        for feats in all_features:
            for t in set(feats):
                df[t] += 1

        if not df:
            max_df = 0
        else:
            max_df = max(df.values())

        idf_all = {}
        for t, freq in df.items():
            if freq < self.min_df:
                continue

            if self.idf_type == 'unnormalized':
                idf_all[t] = 1 + math.log((N + 1) / (freq + 1))
            elif self.idf_type == 'norm_max_df':
                idf_all[t] = 1 + math.log((max_df + 1) / (freq + 1))
            else:
                raise ValueError(f"Unknown idf_type: {self.idf_type}")

        if self.max_features and len(idf_all) > self.max_features:
            top_terms = sorted(idf_all, key=lambda t: df[t], reverse=True)[:self.max_features]
        else:
            top_terms = list(idf_all.keys())

        self.vocab_list_ = sorted(top_terms)
        self.vocabulary_ = {t: i for i, t in enumerate(self.vocab_list_)}
        self.idf_values_ = {t: idf_all[t] for t in self.vocab_list_}
        return self

    def _transform_one(self, text):
        feats = self._extract_features(text)
        tf = self._compute_tf(feats)
        tfidf = {
            self.vocabulary_[t]: tf[t] * self.idf_values_[t]
            for t in tf if t in self.vocabulary_
        }
        return tfidf

    def transform(self, texts):
        V = len(self.vocabulary_)
        N = len(texts)
        X = np.zeros((N, V), dtype=np.float32)
        for i, text in enumerate(texts):
            for idx, val in self._transform_one(text).items():
                X[i, idx] = val
        return X

    def fit_transform(self, texts):
        self.fit(texts)
        return self.transform(texts)


In [6]:
class CustomLogisticRegression:
    """
    Multi-class Logistic Regression (no sklearn).
    Forward:  z = X @ W + b
              P = softmax(z)  (numerically stable)
    Loss:     -mean(log P[correct_class]) + 0.5*l2*||W||^2
    Backward: gradient of loss w.r.t. W, b
    Optimizer: mini-batch SGD
    """

    def __init__(self, num_classes, learning_rate=0.5, num_epochs=30,
                 batch_size=256, l2_reg=1e-4, seed=42):
        self.num_classes   = num_classes
        self.learning_rate = learning_rate
        self.num_epochs    = num_epochs
        self.batch_size    = batch_size
        self.l2_reg        = l2_reg
        self.seed          = seed
        self.W             = None   # (F, C)
        self.b             = None   # (C,)
        self.train_losses  = []
        self.val_losses    = []

    def _softmax(self, Z):
        """Numerically stable softmax. Z: (N,C) -> P: (N,C)"""
        Z_s = Z - np.max(Z, axis=1, keepdims=True)
        exp_Z = np.exp(Z_s)
        return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

    def _loss(self, probs, y):
        """Cross-entropy + L2 regularization."""
        N = len(y)
        log_p = -np.log(np.clip(probs[np.arange(N), y], 1e-12, 1.0))
        return float(np.mean(log_p) + 0.5 * self.l2_reg * np.sum(self.W**2))

    def _gradients(self, X_b, y_b):
        """Compute gradients of loss w.r.t. W and b."""
        N = X_b.shape[0]
        Z  = X_b @ self.W + self.b          # (N, C)
        P  = self._softmax(Z)               # (N, C)
        Y  = np.zeros_like(P)               # one-hot: (N, C)
        Y[np.arange(N), y_b] = 1.0
        dZ = (P - Y) / N                    # (N, C)
        dW = X_b.T @ dZ + self.l2_reg * self.W   # (F, C)
        db = np.sum(dZ, axis=0)             # (C,)
        return dW, db, P

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        """Train using mini-batch SGD."""
        rng = np.random.RandomState(self.seed)
        N, F = X_train.shape
        C = self.num_classes

        # Xavier initialization
        scale = math.sqrt(2.0 / (F + C))
        self.W = rng.randn(F, C).astype(np.float32) * scale
        self.b = np.zeros(C, dtype=np.float32)

        print(f'Training: {N} samples, {F} features, {C} classes')
        print(f'LR={self.learning_rate}, epochs={self.num_epochs}, batch={self.batch_size}')
        print('-' * 70)

        for epoch in range(self.num_epochs):
            idx = rng.permutation(N)
            Xs, ys = X_train[idx], y_train[idx]
            epoch_loss, nb = 0.0, 0

            for start in range(0, N, self.batch_size):
                end = min(start + self.batch_size, N)
                dW, db, P = self._gradients(Xs[start:end], ys[start:end])
                self.W -= self.learning_rate * dW
                self.b -= self.learning_rate * db
                epoch_loss += self._loss(P, ys[start:end])
                nb += 1

            avg_loss = epoch_loss / nb
            self.train_losses.append(avg_loss)

            val_info = ''
            if X_val is not None:
                vp = self.predict_proba(X_val)
                vl = self._loss(vp, y_val)
                va = float(np.mean(np.argmax(vp, axis=1) == y_val))
                self.val_losses.append(vl)
                val_info = f' | Val Loss: {vl:.4f} | Val Acc: {va:.4f}'

            print(f'Epoch {epoch+1:3d}/{self.num_epochs} | Train Loss: {avg_loss:.4f}{val_info}')

        print('-' * 70)
        print('Training complete.')
        return self

    def predict_proba(self, X):
        """Return softmax probabilities. X: (N,F) -> (N,C)"""
        return self._softmax(X @ self.W + self.b)

    def predict(self, X):
        """Return predicted class indices. X: (N,F) -> (N,)"""
        return np.argmax(self.predict_proba(X), axis=1)


print('CustomLogisticRegression defined.')


CustomLogisticRegression defined.


In [7]:
def compute_macro_f1(y_true, y_pred, num_classes):
    """
    Compute Macro-F1 score without sklearn.
    Returns: (macro_f1, per_class_metrics_dict)
    """
    y_true = list(y_true)
    y_pred = list(y_pred)
    assert len(y_true) == len(y_pred)

    TP      = [0] * num_classes
    FP      = [0] * num_classes
    FN      = [0] * num_classes
    support = [0] * num_classes

    for t, p in zip(y_true, y_pred):
        support[t] += 1
        if t == p:
            TP[t] += 1
        else:
            FN[t] += 1
            FP[p] += 1

    per_class = {}
    f1_scores = []
    for c in range(num_classes):
        prec = TP[c] / (TP[c] + FP[c]) if (TP[c] + FP[c]) > 0 else 0.0
        rec  = TP[c] / (TP[c] + FN[c]) if (TP[c] + FN[c]) > 0 else 0.0
        f1   = 2*prec*rec / (prec+rec)  if (prec+rec) > 0       else 0.0
        per_class[c] = {'precision': prec, 'recall': rec, 'f1': f1, 'support': support[c]}
        f1_scores.append(f1)

    macro_f1 = sum(f1_scores) / num_classes
    return macro_f1, per_class


def compute_accuracy(y_true, y_pred):
    """Simple accuracy (no sklearn)."""
    return sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)


print('Evaluation functions defined.')


Evaluation functions defined.


In [8]:
# We will iterate over the 6 combinations, train Logistic Regression, and record results.
# To save time, we will use max_features=10000 and train for 15 epochs per combination.

tf_types = ['unnormalized', 'norm_total_words', 'norm_max_freq']
idf_types = ['unnormalized', 'norm_max_df']

results = []

for tf_t in tf_types:
    for idf_t in idf_types:
        name = f"TF:{tf_t} | IDF:{idf_t}"
        print('=' * 60)
        print(f"Running: {name}")
        print('=' * 60)

        # 1. Vectorize
        vectorizer = CustomTFIDFVectorizer(tf_type=tf_t, idf_type=idf_t, max_features=10000)
        print("Fitting TF-IDF...")
        X_tr = vectorizer.fit_transform(train_texts)
        X_va = vectorizer.transform(val_texts)
        X_te = vectorizer.transform(test_texts)

        # 2. Train
        print("Training Logistic Regression...")
        clf = CustomLogisticRegression(
            num_classes = len(unique_labels),
            learning_rate=0.5,
            num_epochs=15,
            batch_size  = 256,
            l2_reg      = 1e-4,
            seed        = SEED
        )
        # Suppress training epoch outputs to avoid massive logs, just print final
        clf.fit(X_tr, y_train, X_val=X_va, y_val=y_val)

        # 3. Evaluate
        y_te_pred = clf.predict(X_te)
        test_acc = np.mean(y_te_pred == y_test)
        test_f1, _ = compute_macro_f1(y_test, y_te_pred, len(unique_labels))

        print(f"--> Test Acc: {test_acc:.4f} | Test F1: {test_f1:.4f}\n")

        results.append({
            'TF': tf_t,
            'IDF': idf_t,
            'Test_Acc': test_acc,
            'Test_F1': test_f1
        })


Running: TF:unnormalized | IDF:unnormalized
Fitting TF-IDF...
Training Logistic Regression...
Training: 18032 samples, 10000 features, 23 classes
LR=0.5, epochs=15, batch=256
----------------------------------------------------------------------
Epoch   1/15 | Train Loss: 0.8277 | Val Loss: 1.0800 | Val Acc: 0.9139
Epoch   2/15 | Train Loss: 0.3540 | Val Loss: 0.4899 | Val Acc: 0.9366
Epoch   3/15 | Train Loss: 0.2203 | Val Loss: 0.5699 | Val Acc: 0.9241
Epoch   4/15 | Train Loss: 0.1304 | Val Loss: 0.4707 | Val Acc: 0.9343
Epoch   5/15 | Train Loss: 0.0892 | Val Loss: 0.4623 | Val Acc: 0.9348
Epoch   6/15 | Train Loss: 0.0711 | Val Loss: 0.4776 | Val Acc: 0.9330
Epoch   7/15 | Train Loss: 0.0608 | Val Loss: 0.5452 | Val Acc: 0.9295
Epoch   8/15 | Train Loss: 0.0608 | Val Loss: 0.4757 | Val Acc: 0.9308
Epoch   9/15 | Train Loss: 0.0487 | Val Loss: 0.4949 | Val Acc: 0.9326
Epoch  10/15 | Train Loss: 0.0443 | Val Loss: 0.4880 | Val Acc: 0.9335
Epoch  11/15 | Train Loss: 0.0394 | Val Loss

In [9]:
from IPython.display import display, Markdown

md = "| TF Type | IDF Type | Test Accuracy | Test Macro-F1 |\n"
md += "|---|---|---|---|\n"
for r in results:
    md += f"| {r['TF']} | {r['IDF']} | {r['Test_Acc']:.4f} | {r['Test_F1']:.4f} |\n"

display(Markdown("### Final Comparison of TF-IDF Normalization Strategies\n\n" + md))


### Final Comparison of TF-IDF Normalization Strategies

| TF Type | IDF Type | Test Accuracy | Test Macro-F1 |
|---|---|---|---|
| unnormalized | unnormalized | 0.9346 | 0.9350 |
| unnormalized | norm_max_df | 0.9315 | 0.9318 |
| norm_total_words | unnormalized | 0.9337 | 0.9340 |
| norm_total_words | norm_max_df | 0.9324 | 0.9330 |
| norm_max_freq | unnormalized | 0.9346 | 0.9348 |
| norm_max_freq | norm_max_df | 0.9332 | 0.9336 |
